# Venture-Funded Startup Analysis

## Tables

### fs_funded_startups

| Column | Type |
|----------|----------|
| startup_id | STRING |
| startup_name | STRING |
| vc_id | STRING |
| funding | DOUBLE |

### fs_venture_capitalist

| Column | Type |
|----------|----------|
| vc_id | STRING |
| vc_name | STRING |
| funding_limit | DOUBLE |

## Business Rules

- Compute average funding per VC using funding values.
- NULL funding values are excluded from average calculations.
- NULL funding rows still count toward the startup/deal count.
- Only VCs with at least 2 funded startups are eligible.
- Average funding must be strictly greater than funding_limit.
- Return funding_limit and avg_funding rounded to 2 decimal places.
- Sort by avg_funding descending.
- Return only the top 1 VC.

## Output Columns

- avg_funding
- funding_limit
- vc_id
- vc_name

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import *
# fs_funded_startups
fs_funded_startups_schema = StructType([
    StructField("startup_id", StringType(), True),
    StructField("startup_name", StringType(), True),
    StructField("vc_id", StringType(), True),
    StructField("funding", DoubleType(), True)
])

# fs_venture_capitalist
fs_venture_capitalist_schema = StructType([
    StructField("vc_id", StringType(), True),
    StructField("vc_name", StringType(), True),
    StructField("funding_limit", DoubleType(), True)
])

fs_funded_startups_data = [
    ("S1", "Startup 1", "VC1", 2.0),
    ("S2", "Startup 2", "VC1", 1.0),
    ("S3", "Startup 3", "VC2", 2.5),
    ("S4", "Startup 4", "VC2", 2.0),
    ("S5", "Startup 5", "VC3", 1.8),
    ("S6", "Startup 6", "VC3", 1.7)
]

fs_venture_capitalist_data = [
    ("VC1", "VC Firm 1", 1.5),
    ("VC2", "VC Firm 2", 2.0),
    ("VC3", "VC Firm 3", 1.75),
    ("VC4", "VC Firm 4", 2.5)
]

fs_funded_startups = spark.createDataFrame(
    fs_funded_startups_data,
    schema=fs_funded_startups_schema
)

fs_venture_capitalist = spark.createDataFrame(
    fs_venture_capitalist_data,
    schema=fs_venture_capitalist_schema
)

In [0]:
result_df = (
    fs_funded_startups.join(fs_venture_capitalist, on="vc_id")
    .groupBy("vc_id", "vc_name", "funding_limit")
    .agg(count("startup_id").alias("startup_cnt"), avg("funding").alias("avg_funding"))
    .filter((col("avg_funding") > col("funding_limit")) & (col("startup_cnt") >= 2))
    .select(
        "vc_name",
        "vc_id",
        "avg_funding",
        round(col("funding_limit"), 2).alias("funding_limit"),
    )
    .orderBy(col("avg_funding").desc())
    .limit(1)
)

display(result_df)